# 03 · RAG Evaluation
Loads the full 45-question experiment (`src/evaluator.py`, Phase 6) and reviews accuracy, groundedness, and hallucination risk metrics, RAG vs Non-RAG.

In [ ]:
import sys
from pathlib import Path

# Notebooks run from notebooks/, but the src package lives at the project
# root -- add the project root to sys.path so `from src import ...` works.
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

pd.set_option("display.max_colwidth", 100)


In [ ]:
from src import config

results_df = pd.read_csv(config.EXPERIMENT_RESULTS_CSV, encoding='utf-8-sig')
response_metrics = pd.read_csv(config.RESPONSE_METRICS_SUMMARY_CSV, encoding='utf-8-sig')
accuracy_by_language = pd.read_csv(config.ACCURACY_BY_LANGUAGE_CSV, encoding='utf-8-sig')
accuracy_by_category = pd.read_csv(config.ACCURACY_BY_CATEGORY_CSV, encoding='utf-8-sig')
rag_metrics = pd.read_csv(config.RAG_METRICS_SUMMARY_CSV, encoding='utf-8-sig')
risk_distribution = pd.read_csv(config.HALLUCINATION_RISK_DISTRIBUTION_CSV, encoding='utf-8-sig')

print(f'Experiment rows: {len(results_df)}')

## Response metrics (RAG vs Non-RAG)

In [ ]:
response_metrics

## Accuracy by language

In [ ]:
accuracy_by_language

## Accuracy by category

In [ ]:
accuracy_by_category

## RAG metrics summary (groundedness, claim rates, retrieval success)

In [ ]:
rag_metrics

## Hallucination risk distribution

In [ ]:
risk_distribution

## Charts (generated by `src/visualization.py`, Phase 7)

In [ ]:
chart_files = sorted(config.CHARTS_DIR.glob('*.png'))
for chart_path in chart_files:
    display(Image(filename=str(chart_path)))

## Example: inspecting one question end-to-end
A single row from the experiment results, including retrieved context, both responses, and claim-level analysis.

In [ ]:
import json

row = results_df[results_df['question_id'] == 'HIS01_gu'].iloc[0]
print('Question:', row['question'])
print('Reference answer:', row['reference_answer'])
print()
print('Non-RAG response:', row['non_rag_response'])
print('Non-RAG label:', row['non_rag_evaluation_label'])
print()
print('RAG response:', row['rag_response'])
print('RAG label:', row['rag_evaluation_label'])
print()
print('RAG claim analysis:')
for claim in json.loads(row['rag_claim_analysis']):
    print(f"  [{claim['status']}] {claim['claim_text']} (score={claim['similarity_score']})")

## Limitations (read before interpreting the metrics above)

**1. Small, clean, well-separated dataset.** 45 questions across 15 facts is enough to demonstrate the pipeline and produce a real, explainable RAG-vs-Non-RAG gap, but it is not a large-scale benchmark. Near-100% retrieval accuracy and uniformly 'Low' hallucination risk reflect this dataset's deliberate design (topically distinct facts, short answers), not proof the underlying methods are error-free on harder or noisier data.

**2. Correctness checking is a lexical heuristic.** `evaluate_correctness()` uses normalized text matching (exact match for Strict, ~60% significant-word overlap for Relaxed) against the reference answer -- not semantic understanding. It will not catch every valid paraphrase and can be fooled by superficial word overlap in principle, though this did not appear to be an issue on this dataset's short, unambiguous answers.

**3. Hallucination detection is a semantic similarity heuristic, not entailment.** A claim scoring above `CLAIM_SUPPORTED_THRESHOLD` is semantically close in wording to some retrieved passage -- this is not verified logical entailment. A claim scoring low is **unsupported by the retrieved context**, which is explicitly *not* the same as being **factually incorrect** -- these two dimensions are tracked independently throughout this project and should never be conflated when reading results.

**4. Thresholds were calibrated on the 'supported' end of the distribution.** `src/calibrate_thresholds.py` measured similarity scores for RAG claims generated directly from their own retrieved context (a near-best-case scenario), confirming genuinely-grounded claims score well above `CLAIM_SUPPORTED_THRESHOLD = 0.80`. This dataset did not naturally produce many low-similarity claims, so the Medium/High hallucination-risk buckets are implemented and correct but not heavily stress-tested here.

## Future improvements
- Expand the dataset with harder, more ambiguous, or intentionally conflicting facts to stress-test the Medium/High hallucination-risk buckets.
- Add an NLI-based entailment check for English as a secondary signal alongside the similarity heuristic (with an honest note that Hindi/Gujarati would remain similarity-only).
- Introduce adversarial or out-of-domain questions to test retrieval failure modes and observe how groundedness degrades.